# 🧩 Mini-Lab: Schema Validation with Pydantic

**Module 3: Structured Extraction** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Use** JSON mode for guaranteed valid JSON output
2. **Define** Pydantic models for type-safe extraction
3. **Handle** nested structures and optional fields
4. **Validate** extracted data automatically

## Target Concepts

| Concept | Description |
|---------|-------------|
| JSON mode | OpenAI API feature that guarantees valid JSON output format |
| Pydantic models | Python data validation library using type annotations |
| Type validation | Automatic checking that data matches expected types (str, int, float, etc.) |
| Nested schemas | Pydantic models containing other models for complex hierarchical data |

In [ ]:
import json
from pathlib import Path
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
from dotenv import load_dotenv
from IPython.display import Markdown, display

def md(text):
    display(Markdown(text))

load_dotenv()
client = OpenAI()

# Load sample data
DATA_DIR = Path('../data/')
sample_invoice = (DATA_DIR / 'invoices' / 'invoice_001_corporate.txt').read_text(encoding='utf-8')

print('✓ Setup complete')
print(f'\n📄 Sample invoice loaded ({len(sample_invoice)} chars)')

---
## 1. The Problem: Invalid JSON

Without JSON mode, LLMs sometimes return invalid JSON or include extra text.

In [ ]:
# This might fail or return invalid JSON
def extract_without_json_mode(document: str) -> str:
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'user', 'content': f'Extract invoice data as JSON:\n{document}'}
        ],
        temperature=0
    )
    return response.choices[0].message.content

result = extract_without_json_mode(sample_invoice)
print('Raw response (first 500 chars):')
print(result[:500])

# Try to parse
try:
    parsed = json.loads(result)
    print('\n✅ Valid JSON')
except json.JSONDecodeError as e:
    print(f'\n❌ Invalid JSON: {e}')

---
## 2. JSON Mode: Guaranteed Valid JSON

OpenAI's **JSON mode** guarantees the response is valid JSON.

In [ ]:
def extract_with_json_mode(document: str) -> dict:
    """
    Extract data with guaranteed valid JSON output.
    
    Note: The prompt MUST mention 'JSON' for JSON mode to work.
    """
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': 'Extract invoice data and return as JSON.'
            },
            {
                'role': 'user',
                'content': f'Extract from this invoice:\n{document}'
            }
        ],
        response_format={'type': 'json_object'},  # Enable JSON mode
        temperature=0
    )
    
    # Guaranteed to be valid JSON
    return json.loads(response.choices[0].message.content)

result = extract_with_json_mode(sample_invoice)

print('✅ JSON mode result:')
print(json.dumps(result, indent=2)[:1000])

---
## 3. Pydantic Models: Type-Safe Extraction

**Pydantic** provides:
- Type validation
- Default values
- Field descriptions
- Automatic JSON schema generation

In [ ]:
# Define a simple invoice schema
class SimpleInvoice(BaseModel):
    """Basic invoice data."""
    invoice_number: str = Field(..., description="Invoice ID/number")
    invoice_date: str = Field(..., description="Date issued")
    vendor_name: str = Field(..., description="Company issuing invoice")
    customer_name: str = Field(..., description="Company being billed")
    total: float = Field(..., description="Total amount due")

# View the generated JSON schema
print('📋 Generated JSON Schema:')
print(json.dumps(SimpleInvoice.model_json_schema(), indent=2))

In [ ]:
def extract_with_pydantic(document: str, model: type[BaseModel]) -> BaseModel:
    """
    Extract data and validate with Pydantic.
    
    Args:
        document: Document text
        model: Pydantic model class
        
    Returns:
        Validated Pydantic model instance
    """
    schema = json.dumps(model.model_json_schema(), indent=2)
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {
                'role': 'system',
                'content': f'''Extract data according to this JSON schema:
{schema}

Return only valid JSON matching the schema.'''
            },
            {'role': 'user', 'content': f'Document:\n{document}'}
        ],
        response_format={'type': 'json_object'},
        temperature=0
    )
    
    data = json.loads(response.choices[0].message.content)
    return model(**data)  # Pydantic validation happens here

# Extract and validate
invoice = extract_with_pydantic(sample_invoice, SimpleInvoice)

print('✅ Extracted and validated:')
print(f'  Invoice #: {invoice.invoice_number}')
print(f'  Vendor: {invoice.vendor_name}')
print(f'  Customer: {invoice.customer_name}')
print(f'  Total: ${invoice.total:,.2f}')
print(f'\n  Type of result: {type(invoice).__name__}')

---
## 4. Optional Fields and Defaults

Real documents often have missing fields. Use `Optional` for nullable fields.

In [ ]:
class InvoiceWithOptionals(BaseModel):
    """Invoice with optional fields."""
    # Required fields
    invoice_number: str
    vendor_name: str
    total: float
    
    # Optional fields (can be null)
    invoice_date: Optional[str] = None
    due_date: Optional[str] = None
    customer_name: Optional[str] = None
    vendor_email: Optional[str] = None
    vendor_phone: Optional[str] = None
    tax_amount: Optional[float] = None
    payment_status: Optional[str] = None

# Test with a minimal document
minimal_doc = '''Invoice #12345
From: Acme Corp
Total Due: $500'''

result = extract_with_pydantic(minimal_doc, InvoiceWithOptionals)

print('📄 Extraction from minimal document:')
print(f'  Invoice #: {result.invoice_number}')
print(f'  Vendor: {result.vendor_name}')
print(f'  Total: ${result.total}')
print(f'  Due Date: {result.due_date}')  # Will be None
print(f'  Tax: {result.tax_amount}')  # Will be None

---
## 5. Nested Schemas

For complex data like line items, use nested Pydantic models.

In [ ]:
class LineItem(BaseModel):
    """A single line item."""
    description: str = Field(..., description="Item description")
    quantity: Optional[float] = Field(None, description="Quantity")
    unit_price: Optional[float] = Field(None, description="Price per unit")
    amount: float = Field(..., description="Line total")


class DetailedInvoice(BaseModel):
    """Complete invoice with line items."""
    invoice_number: str
    invoice_date: str
    vendor_name: str
    customer_name: str
    
    line_items: list[LineItem] = Field(
        default_factory=list,
        description="List of items/services"
    )
    
    subtotal: float
    tax_amount: Optional[float] = None
    total: float


# Extract with nested schema
detailed = extract_with_pydantic(sample_invoice, DetailedInvoice)

print('📋 Detailed Invoice Extraction:')
print(f'Invoice: {detailed.invoice_number}')
print(f'From: {detailed.vendor_name}')
print(f'To: {detailed.customer_name}')
print(f'\nLine Items ({len(detailed.line_items)}):')

for i, item in enumerate(detailed.line_items[:5], 1):  # Show first 5
    qty = f"x{item.quantity}" if item.quantity else ""
    print(f'  {i}. {item.description[:40]} {qty} = ${item.amount:,.2f}')

print(f'\nSubtotal: ${detailed.subtotal:,.2f}')
if detailed.tax_amount:
    print(f'Tax: ${detailed.tax_amount:,.2f}')
print(f'Total: ${detailed.total:,.2f}')

---
## 6. Validation Errors

Pydantic catches data issues automatically.

In [ ]:
# Demonstrate validation error
try:
    # Try to create invoice with wrong types
    bad_invoice = SimpleInvoice(
        invoice_number="123",
        invoice_date="2024-01-01",
        vendor_name="Test",
        customer_name="Customer",
        total="not a number"  # Should be float!
    )
except ValidationError as e:
    print('❌ Validation Error:')
    for error in e.errors():
        print(f"  Field '{error['loc'][0]}': {error['msg']}")

In [ ]:
# Safe extraction with error handling
def safe_extract(document: str, model: type[BaseModel]) -> tuple[BaseModel | None, str | None]:
    """
    Extract with error handling.
    
    Returns:
        Tuple of (result, error_message)
    """
    try:
        result = extract_with_pydantic(document, model)
        return result, None
    except ValidationError as e:
        return None, f"Validation error: {e}"
    except json.JSONDecodeError as e:
        return None, f"JSON parse error: {e}"
    except Exception as e:
        return None, f"Extraction error: {e}"

# Test safe extraction
result, error = safe_extract(sample_invoice, SimpleInvoice)

if result:
    print(f'✅ Success: Invoice #{result.invoice_number}')
else:
    print(f'❌ Failed: {error}')

---
## 7. Using Our Project Schemas

Let's use the schemas defined in `schemas.py`.

In [ ]:
from document_schemas import Invoice, Receipt, Resume, Contract

# Extract using the full Invoice schema
full_invoice = extract_with_pydantic(sample_invoice, Invoice)

print('📋 Full Invoice Schema Extraction:')
print(f'Invoice #: {full_invoice.invoice_number}')
print(f'Date: {full_invoice.invoice_date}')
print(f'Due: {full_invoice.due_date}')
print(f'Vendor: {full_invoice.vendor_name}')
print(f'Customer: {full_invoice.customer_name}')
print(f'Items: {len(full_invoice.line_items)}')
print(f'Subtotal: ${full_invoice.subtotal:,.2f}')
print(f'Tax: ${full_invoice.tax_amount:,.2f}' if full_invoice.tax_amount else 'Tax: N/A')
print(f'Total: ${full_invoice.total:,.2f}')
print(f'Status: {full_invoice.payment_status}')

## 🎯 Summary

### Key Takeaways

1. **JSON mode** — guarantees valid JSON output by setting `response_format={'type': 'json_object'}`
2. **Pydantic validation** — automatically validates types, required fields, and nested structures
3. **Optional fields** — use `Optional[T]` for fields that may be missing in documents
4. **Nested models** — define complex hierarchical data like line items using nested Pydantic classes
5. **Schema-driven extraction** — pass JSON schema to LLM to guide extraction format

---
## Next Steps

In the next notebook, we'll learn how to:
- **Detect document types** automatically
- **Select the right schema** based on document type
- **Build a universal extractor** for all document types

→ Continue to **mini-doc-extractor-03-multi-document-types.ipynb**